### import


In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mp
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

### README

In [ ]:
with open('./raw/m1/README', 'r') as fp:
  readme = fp.read()

In [ ]:
print(readme)

### 데이터 가져오기

오류발생 =>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 3114: invalid continuation byte  

해결방법)
- 1)  0xe9는 UTF-8에서 유효한 단일 바이트 문자가 아니다.
- 2) 0xe9 바이트는 ISO-8859-1(European Latin-1)혹은 cp1252(윈도우 인코딩)로 설정해 준다.

In [ ]:
movies = pd.read_csv('./raw/m1/movies.dat', delimiter='::', header=None, engine='python', encoding='ISO-8859-1' )


In [ ]:
print(movies.shape)

In [ ]:
#MovieID::Title::Genres
movies.head()

In [ ]:
movies.sample(5)

In [ ]:
movies.info()

In [ ]:
movies.set_index(0)  # default : inplace=False

In [ ]:
movies.head()

In [ ]:
users = pd.read_csv('./raw/m1/users.dat', sep='::', engine='python', header=None)

In [ ]:
#UserID::Gender::Age::Occupation::Zip-code
users.head()

In [ ]:
users.info()

In [ ]:
users.shape

In [ ]:
ratings = pd.read_csv('./raw/m1/ratings.dat', sep='::', engine='python', header=None)


In [ ]:
#UserID::MovieID::Rating::Timestamp
ratings.head()

In [ ]:
ratings.info()

In [ ]:
ratings.shape

### 데이터 전처리(preprocessing)

In [ ]:
print(type(movies))

In [ ]:
print(dir(movies))

In [ ]:
#MovieID::Title::Genres
movies.columns

In [ ]:
# 버전에 따라서는 컬럼명이 숫자일 때 문자열로 써야하는 경우도 있다.
movies.rename(columns={0:'MovieID', 1:'Title', 2:'Genres'}, inplace=True)

In [ ]:
movies.head()

In [ ]:
#UserID::Gender::Age::Occupation::Zip-code

# 데이터를 읽어올 때 컬럼명을 지정할 수 있다.
users = pd.read_csv('./raw/m1/users.dat', sep='::', engine='python',header=None,
                   names=['UserID','Gender','Age','Occupation','Zip-code'])

In [ ]:
users.head()

In [ ]:
#UserID::MovieID::Rating::Timestamp

#ratings.rename(columns={0:'UserID', 1:'MovieID', 2:'Rating', 3:'Timestamp'}, inplace=True)
#ratings.rename(mapper={0:'UserID', 1:'MovieID', 2:'Rating', 3:'Timestamp'}, axis=1, inplace=True )

ratings = pd.read_csv('./raw/m1/ratings.dat', sep='::', engine='python', header=None,
                      names=['UserID','MovieID','Rating','Timestamp'])

In [ ]:
ratings.head()

### 데이터 구조 변경

merge와 join 비교

In [ ]:
df = pd.DataFrame({'key':['K0','K1','K2','K3','K4','K5'],
                   'A' : ['A0','A1','A2','A3','A4','A5']})
df

In [ ]:
other = pd.DataFrame({'key':['K0','K1','K2'],
                       'B' : ['B0','B1','B2']})
other

In [ ]:
# 공통적인 key값이 있는 것만 병합이 된다.
# how의 속성값이 inner이다.
df.merge(other)

In [ ]:
df.head(2)

In [ ]:
other.head(2)

In [ ]:
# join()은 속성의 값이 left(how:str='left')이므로 left outer join을 실행한다.
# left outer join
df.join(other.set_index('key'), on='key')

In [ ]:
# Left Outer join
other.join(df.set_index('key'), on='key')

In [ ]:
# Right Outer Join
other.join(df.set_index('key'), on='key', how='right')

In [ ]:
# Inner Join
other.join(df.set_index('key'), on='key', how='inner')

merge는 이름이 같은 column을 기준으로 데이터를 합친다.

In [ ]:
# 관람한 영화들에 대해서만 merge를 합다.
data = ratings.merge(users).merge(movies)
data.head()

In [ ]:
data.columns

In [ ]:
#recommendation_data = data.loc[:, 'UserID':'Rating']
#recommendation_data = data.iloc[:, 0:3]
recommendation_data = data[['UserID', 'MovieID', 'Rating']]

In [ ]:
recommendation_data.head(3)

In [ ]:
recommendation_data.shape

pivot

In [ ]:
recommendation_pivot = recommendation_data.pivot(index ='UserID', columns='MovieID', values='Rating')
recommendation_pivot.head()

In [ ]:
recommendation_pivot.sample(5)

missing data처리  
- drop이라는 이름이 붙으면 버리는 기능이다.
- fill이라는 이름이 붙으면 채우는 기능이다.


In [ ]:
recommendation_pivot.fillna(0, inplace=True)

In [ ]:
recommendation_pivot.sample(5)

stack, unstack
 - MultiIndex을 만든 다음, unstack으로 pivot과 똑같은 결과를 만들 수 있다.

In [ ]:
recommendation_data.head()

In [ ]:
#set_index에 이름을 여러개 넣으면 MultiIndex가 된다.
data_stack = recommendation_data.set_index(['UserID', 'MovieID']).stack().copy()
data_stack

In [ ]:
data_unstack = recommendation_data.set_index(['UserID', 'MovieID']).unstack().copy()

In [ ]:
data_unstack.head()

In [ ]:
data_unstack.fillna(0, inplace=True)

In [ ]:
data_unstack.head()

### 데이터 분석

In [ ]:
recommendation_pivot.head(3)

In [ ]:
print(recommendation_pivot.loc[2746])
print(recommendation_pivot.loc[2746].value_counts())

In [ ]:
# UserID가 2746인 사람이 본 영화 목록
mov_user_2746 = recommendation_pivot.loc[2746]
# print(mov_user_2746)

# mov_user_2746 Series에서 평점이 0보다 큰 값만 가져온다.
mov_user_2746[mov_user_2746>0].head()


In [ ]:
# UserId 2746인 사람이 매긴 평점별 영화 개수
mov_user_2746[mov_user_2746>0].value_counts()

In [ ]:
mov_user_2746[mov_user_2746>0].unique()

In [ ]:
# 함수를 정의해서 일반화 시킨다.
def movie_seen(user_id):
  return recommendation_pivot.loc[user_id][recommendation_pivot.loc[user_id]>0]

In [ ]:
# 7번 유저가 본 영화
movie_seen(7)

corr
 - corr은 column기준으로 상관관계를 분석해 준다.

In [ ]:
# corr : correlationship(상관관계)
users[['UserID', 'Age']].corr()

In [ ]:
recommendation_pivot.head()

In [ ]:
# UserID을 기준으로 상관관계를 구할 것이므로
# column이 UserID여야 한다.
small_test = recommendation_pivot.T.iloc[:500,:500]
small_test

In [ ]:
small_test_corr = small_test.corr()
small_test_corr

UserID가 5인 사람과 상관관계 높은 상위 10명을 뽑을 수 있다.

In [ ]:
# sort_values( )  값의 크기순으로 데이터 순서를 정렬한다.
# ascending=True은 오름차순이다. default로 설정되여 있다.
# ascending=False은 내림차순이다.

# 0 인덱스의 값은 자기 자신을 의미하므로 제외하고 1에서 부터 11미만의 값을 가져온다.
small_test_corr[5].sort_values(ascending=False)[1:11]

In [ ]:
# 함수 정의를 통한 일반화
def nearest_user(user_id, n):
  return small_test_corr.loc[user_id].sort_values(ascending =False)[1: n+1]



In [ ]:
# 5번 유저와 상관관계 높은 3명을 뽑아 올 수 있다.
nearest_user(5, 3)

상관관계가 높은 193번과 5번 유저가 본 영화를 비교한다.

In [ ]:
# 193번 유저가 본 영화를 가져온다.
movie_seen(193)

In [ ]:
# 5번 유저가 본 영화를 가져온다.
movie_seen(5)

In [ ]:
# 193번은 보고, 5번은 보지 않은 영화 목록(MovieID)을 뽑아온다.
unseen = set(movie_seen(193).index)- set(movie_seen(5).index)
len(unseen)

In [ ]:
unseen_movie = recommendation_data[recommendation_data.MovieID.isin(unseen)]
unseen_movie

In [ ]:
# 193번 유저가 5점 준 영화
unseen_movie[(unseen_movie.Rating==5) & (unseen_movie.UserID==193)]

In [ ]:
# 추천할 영화제목을 가져온다.
unseen_movie[(unseen_movie.Rating==5) & (unseen_movie.UserID==193)].merge(movies)

1. 193번은 보고 5번은 안본 영화
2. recommendation_data에서 위의 1번 결과에 해당하는 목록을 가져옴   
3. Rating이 5점이고 193이 본 영화 목록을 가져와서 영화제목과 장르를 연결

In [ ]:
#user_id(193)
#other_id(5)

def unseen_movie(user_id, other_id):
  unseen = set(movie_seen(user_id).index)- set(movie_seen(other_id).index)
  unseen_movie = recommendation_data[recommendation_data.MovieID.isin(unseen)]
  return unseen_movie[(unseen_movie.Rating==5) & (unseen_movie.UserID==user_id)].merge(movies)

In [ ]:
# 193번은 보고 5번은 보지 않은 영화중에 평점 5인 영화를 유저 5번에 추천
unseen_movie(193, 5)

7번과 유사한 2명 뽑아서 둘 중 한명이라고 평점이 5점인 영화 추천

In [ ]:
# 유저7과 유사한 2명 가져옴  (288,251)
def nearest_user(user_id, n):
   return  small_test_corr.loc[user_id].sort_values(ascending=False)[1:n+1]


def movie_seen(user):
   return recommendation_pivot.loc[user][recommendation_pivot.loc[user]>0]


In [ ]:
movie_seen(288)

In [ ]:
nearest_user(7, 2)

In [ ]:
nearest_user(7, 2).shape

In [ ]:
# 각각의 유저는 user0, user1로 해준다.
user0 = nearest_user(7, 2).index[0]  #288
user1 = nearest_user(7,2).index[1]   #251
print(user0, user1)

In [ ]:
userlist =  nearest_user(7, 2).index
print(userlist[0])
print(userlist[1])

In [ ]:
movie_seen(user0)

In [ ]:
# user0(288)본 MovieID을 가져온다.
movie_seen(user0).index

In [ ]:
# user1(251)본 MovieID을 가져온다.
movie_seen(user1).index

In [ ]:
# user0(288)과 user1(251)이 본 영화중 5점인 영화 MovieID을 가져온다.

def recommend_movie(user_id, n):
  user_list = nearest_user(user_id, n).index

  #user0(288)과 user1(251)
  user_mv_list = recommendation_data[(recommendation_data.UserID.isin(user_list)) & (recommendation_data.Rating==5)]
  print(user_mv_list.columns)

  # UserID 7번이 본 영화목록
  user7_mv_list = movie_seen(user_id)

  # UserID 288번이나  251번이  본 영화이지만 7번은 보지 않는 영화
  unseen_list = set(user_mv_list['MovieID']) - set(user7_mv_list.index)

  return movies[movies['MovieID'].isin(unseen_list)].reset_index(drop=True)


In [ ]:
test = recommend_movie(7,2)
print(test)

In [ ]:
user7_mv_list = movie_seen(7).index
print(user7_mv_list)

In [ ]:
len((set(test['MovieID']) - set(movie_seen(7).index)))

In [ ]:
len(test['MovieID'])

In [ ]:
%%writefile movie_system.py

import numpy as np
import pandas as pd
import matplotlib as mp
import matplotlib.pyplot as plt
import seaborn as sns
import platform


# OS에 따른 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')  # 윈도우: 맑은 고딕
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')    # 맥: 애플 고딕
else:
    plt.rc('font', family='NanumBarunGothic') # 리눅스

# 마이너스 기호 깨짐 방지
plt.rc('axes', unicode_minus=False)

#파일 로딩
def data_load():
  movies = pd.read_csv('./raw/m1/movies.dat', delimiter='::', header=None, engine='python', encoding='ISO-8859-1',
                    names=['MovieID', 'Title', 'Genres'] )
  users = pd.read_csv('./raw/m1/users.dat', sep='::', engine='python',header=None,
                    names=['UserID','Gender','Age','Occupation','Zip-code'])
  ratings = pd.read_csv('./raw/m1/ratings.dat', sep='::', engine='python', header=None,
                    names=['UserID','MovieID','Rating','Timestamp'])
  return movies, users, ratings

# merge
def data_merge(movies, users, ratings):
  data = ratings.merge(users).merge(movies)
  recommendation_data = data[['UserID', 'MovieID', 'Rating']]
  return recommendation_data

#  pivot, corr
def data_pivot_corr(recommendation_data):
  recommendation_pivot = recommendation_data.pivot(index ='UserID', columns='MovieID', values='Rating')
  recommendation_pivot.fillna(0, inplace=True)
  return recommendation_pivot

def nearest_user(small_test_corr, user_id, n):
  return small_test_corr.loc[user_id].sort_values(ascending =False)[1: n+1]

def movie_seen(user_id):
   return recommendation_pivot.loc[user_id][recommendation_pivot.loc[user_id]>0]

def recommend_movie(recommendation_pivot, movies, user_id, n):
  small_test_corr = recommendation_pivot.T.iloc[:500,:500].corr()
  user_list = nearest_user(small_test_corr, user_id, n).index
  user_mv_list = recommendation_data[(recommendation_data.UserID.isin(user_list)) & (recommendation_data.Rating==5)]
  user7_mv_list = movie_seen(user_id)
  unseen_list = set(user_mv_list['MovieID']) - set(user7_mv_list.index)
  return movies[movies['MovieID'].isin(unseen_list)].reset_index(drop=True)

if __name__  == '__main__':
  movies, users, ratings = data_load()
  recommendation_data = data_merge(movies, users, ratings)
  recommendation_pivot = data_pivot_corr(recommendation_data)
  userid =  int(input('UserId 입력:'))
  movie_receive = recommend_movie(recommendation_pivot, movies,  userid, 2)
  print(len(movie_receive))
  print(movie_receive)

In [ ]:
!pwd

In [ ]:
# !python movie_system.py

streamlit 생성

In [ ]:
# !uv add streamlit

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os


# 데이터 로딩
@st.cache_data
def data_load():
    movies = pd.read_csv('./raw/m1/movies.dat', delimiter='::', header=None, engine='python', encoding='ISO-8859-1',
                         names=['MovieID', 'Title', 'Genres'])
    users = pd.read_csv('./raw/m1/users.dat', sep='::', engine='python', header=None,
                        names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'])
    ratings = pd.read_csv('./raw/m1/ratings.dat', sep='::', engine='python', header=None,
                          names=['UserID', 'MovieID', 'Rating', 'Timestamp'])
    return movies, users, ratings

# merge
@st.cache_data
def data_merge(movies, users, ratings):
    data = ratings.merge(users).merge(movies)
    recommendation_data = data[['UserID', 'MovieID', 'Rating']]
    return data, recommendation_data

# pivot
@st.cache_data
def data_pivot_corr(recommendation_data):
    pivot = recommendation_data.pivot(index='UserID', columns='MovieID', values='Rating')
    pivot.fillna(0, inplace=True)
    return pivot

# 유사 사용자
def nearest_user(corr_matrix, user_id, n):
    return corr_matrix.loc[user_id].sort_values(ascending=False)[1:n+1]

# 본 영화 목록
def movie_seen(recommendation_pivot, user_id, movies):
    seen = recommendation_pivot.loc[user_id][recommendation_pivot.loc[user_id] > 0]
    return movies[movies['MovieID'].isin(seen.index)].assign(MyRating=seen.values)

# 추천
def recommend_movie(pivot, data, movies, user_id, n=2):
    corr = pivot.T.iloc[:500, :500].corr()
    similar_users = nearest_user(corr, user_id, n).index
    sim_user_corr = nearest_user(corr, user_id, n)
    similar_data = data[(data.UserID.isin(similar_users)) & (data.Rating == 5)]
    seen = pivot.loc[user_id][pivot.loc[user_id] > 0]
    unseen = set(similar_data['MovieID']) - set(seen.index)
    return movies[movies['MovieID'].isin(unseen)].reset_index(drop=True), sim_user_corr



# main
def main():

    st.title("사용자 기반 영화 추천 시스템")
    st.markdown("**유사 사용자 기반 협업 필터링**으로 추천합니다.")


    with st.spinner("데이터 로딩 중..."):
        movies, users, ratings = data_load()
        full_data, recommendation_data = data_merge(movies, users, ratings)
        pivot = data_pivot_corr(recommendation_data)

    user_id = st.selectbox("사용자 선택", pivot.index.tolist())
    top_n = st.slider("추천받을 영화 수", 1, 10, 3)

    if st.button("영화 추천받기"):
        with st.spinner("추천 처리 중..."):
            recommended_movies, sim_user_corr = recommend_movie(pivot, recommendation_data, movies, user_id, top_n)
            seen_movies = movie_seen(pivot, user_id, movies)

        st.subheader("내가 본 영화 목록")
        st.dataframe(seen_movies[['Title', 'Genres', 'MyRating']].sort_values('MyRating', ascending=False))

        st.subheader("유사 사용자 Top-N")
        st.table(pd.DataFrame({
            "UserID": sim_user_corr.index,
            "상관계수": sim_user_corr.values
        }))

        st.subheader("유사 사용자 상관계수 시각화")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(x=sim_user_corr.values, y=sim_user_corr.index, ax=ax)
        ax.set_xlabel("상관계수")
        ax.set_ylabel("UserID")
        ax.set_title("Top-N 유사 사용자 상관계수")
        st.pyplot(fig)

        st.subheader("추천 영화 목록")
        st.dataframe(recommended_movies[['Title', 'Genres']])

if __name__ == '__main__':
    main()